## Data Prep for RAG (Unstructured Data)

![data_prep](./Assets/data_prep.png)

### Installing Utilities and Libraries

In [ ]:
%pip install numpy==1.26.4 openai==1.69.0 langchain-community==0.4.1 PyPDF2==3.0.1

### Restarting our Python Environment

In [ ]:
dbutils.library.restartPython()

### Saving Knowledge Base Docs to UC Volume and Local File System

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS genai_lab

In [ ]:
# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

In [ ]:
import os

source_folder = "./knowledge_base"
dbfs_target_folder = f"/Volumes/{catalog_name}/default/genai_lab/tmp/knowledge_base"

os.makedirs(source_folder, exist_ok=True)
dbutils.fs.mkdirs(dbfs_target_folder)

subfolders = ['docs']
for subfolder in subfolders:
    local_subfolder = os.path.join(source_folder, subfolder)
    dbfs_subfolder = f"{dbfs_target_folder}/{subfolder}"
    os.makedirs(local_subfolder, exist_ok=True)
    dbutils.fs.mkdirs(dbfs_subfolder)
    for filename in os.listdir(local_subfolder):
        src = os.path.join(local_subfolder, filename)
        dst = f"{dbfs_subfolder}/{filename}"
        if os.path.isfile(src) and os.path.getsize(src) > 0:
            dbutils.fs.cp(f"file:{os.path.abspath(src)}", dst, recurse=False)

### Verify Data is Stored in Local File System

In [ ]:
print(os.listdir("./knowledge_base/"))
print(" \n docs: ")
print(os.listdir("./knowledge_base/docs/"))

### Verify Data is Stored in UC Volume

In [ ]:
print(dbutils.fs.ls(f"/Volumes/{catalog_name}/default/genai_lab/tmp/knowledge_base/"))
print(" \n docs: ")
print(dbutils.fs.ls(f"/Volumes/{catalog_name}/default/genai_lab/tmp/knowledge_base/docs/"))

### Creating the RAG Table Schema in Unity Catalog

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS YOUR_UNITY_CATALOG_NAME.RAG

### Extracting PDF Content and Storing as Table with Langchain Chunking Strategy

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from PyPDF2 import PdfReader

def perform_fixed_size_chunking(document, chunk_size=2000, chunk_overlap=500):
    """
    Performs recursive chunking on a document with specified overlap.
    Uses RecursiveCharacterTextSplitter which tries multiple separators.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    return text_splitter.split_text(document)

import os

docs_folder = "./knowledge_base/docs"
dbfs_docs_folder = f"/Volumes/{catalog_name}/default/genai_lab/tmp/knowledge_base/docs"
all_docs = []

for filename in os.listdir(docs_folder):
    file_path = os.path.join(docs_folder, filename)
    dbfs_path = f"{dbfs_docs_folder}/{filename}"
    if os.path.isfile(file_path) and filename.lower().endswith(".pdf"):
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            text = ""
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text
            if text.strip():
                chunks = perform_fixed_size_chunking(text)
                for i, chunk in enumerate(chunks):
                    if chunk.strip():
                        all_docs.append({
                            "content_path": dbfs_path,
                            "chunk": chunk
                        })

if all_docs:
    df = spark.createDataFrame(all_docs)
    print(f"Total chunks created: {df.count()}")
    print(f"\nChunks per document:")
    df.groupBy("content_path").count().show(truncate=False)
    display(df)
else:
    print("No chunks extracted from documents.")

In [ ]:
df.write.mode("overwrite").saveAsTable("RAG.docs_chunks")

### Creating the Final Multi-Modal RAG Table in Unity Catalog

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE RAG.final_rag_dataset AS
SELECT monotonically_increasing_id() AS id, content_path, chunk FROM RAG.docs_chunks
""")

display(spark.table("RAG.final_rag_dataset"))